# 01 - Penyelarasan Berita dengan JISDOR

Input: `data/processed/cnbc_articles_clean.csv` (hasil cleaning) dan `data/raw/Informasi Kurs Jisdor.xlsx`.

Output:
- `data/processed/articles_aligned.csv`: tiap artikel ditempelkan ke satu hari bursa (`trading_date`).
- `data/processed/daily_kurs.csv`: satu baris per hari bursa berisi kurs, return, target, dan fitur lag kurs.

Berita jalan 24/7, kurs hanya ada di hari kerja. Jadi pertama-tama tiap artikel harus ditentukan "ikut" hari bursa yang mana.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

def find_repo_root():
    """Cari folder root repo (yang berisi data/ dan src/), dari mana pun notebook dijalankan."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"Root repo (berisi data/ dan src/) tidak ditemukan dari {here}")

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
SPLIT_DIR = ROOT / "data" / "split"
for d in (PROCESSED_DIR, SPLIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

ARTICLES_FILE = PROCESSED_DIR / "cnbc_articles_clean.csv"
JISDOR_FILE = RAW_DIR / "Informasi Kurs Jisdor.xlsx"   # boleh .xlsx atau .csv
DATE_COL, RATE_COL = "Tanggal", "Kurs"                 # sesuaikan dengan nama kolom di file

# Jam cutoff (WIB) = kapan kurs hari itu ditetapkan. 16:00 WIB (jam tutup pasar) sesuai
# docs/temporal_alignment_strategy.md dan src/jisdor/temporal_alignment.py (sudah dipakai
# dan didokumentasikan tim di laporan Tugas 1). Berita yang terbit setelah jam ini dihitung
# untuk hari bursa berikutnya.
CUTOFF_HOUR, CUTOFF_MINUTE = 16, 0

START_DATE, END_DATE = "2021-09-01", "2026-09-01"

for f in (ARTICLES_FILE, JISDOR_FILE):
    if not f.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {f}")
print("Root repo:", ROOT)

## 1. Baca artikel

`published_wib` dibaca ulang sebagai waktu WIB. Aturan penyelarasan bergantung pada jam terbit, jadi kolom ini yang dipakai (bukan tanggalnya saja).

In [ ]:
arts = pd.read_csv(ARTICLES_FILE)
arts["published_wib"] = pd.to_datetime(arts["published_wib"], utc=True).dt.tz_convert("Asia/Jakarta")
arts = arts.sort_values("published_wib").reset_index(drop=True)

print(arts.shape)
print(arts["published_wib"].min(), "->", arts["published_wib"].max())

## 2. Baca JISDOR

Format dari BI bisa memakai bulan berbahasa Indonesia ("Agustus", "Mei") dan angka bergaya Indonesia (`16.100,50`), jadi parsernya dibuat toleran. Ada pengecekan rentang kurs supaya salah baca langsung ketahuan, dan tanggal ganda dibuang (yang terakhir dipakai).

In [ ]:
ID_MONTHS = {
    "januari": "January", "februari": "February", "maret": "March", "mei": "May",
    "juni": "June", "juli": "July", "agustus": "August", "oktober": "October",
    "desember": "December", "agu": "Aug", "agt": "Aug", "okt": "Oct", "des": "Dec",
    "mar": "Mar", "mei": "May", "nop": "Nov",
}

def parse_dates(series):
    """Ubah kolom tanggal jadi datetime. Urutan hari/bulan dideteksi dari seluruh kolom."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series)
    s = series.astype(str).str.strip().str.lower()
    for k, v in ID_MONTHS.items():
        s = s.str.replace(rf"\b{k}\b", v, regex=True)

    iso = pd.to_datetime(s, format="ISO8601", errors="coerce")
    if iso.notna().mean() > 0.95:                       # 2025-01-31
        return iso

    parts = s.str.extract(r"^(\d{1,2})/(\d{1,2})/(\d{4})").apply(pd.to_numeric)
    if parts.notna().all(axis=1).mean() > 0.95:         # 31/01/2025 atau 1/31/2025
        day_first = bool((parts[0] > 12).any() and not (parts[1] > 12).any())
        print("Format tanggal terdeteksi:", "hari/bulan/tahun" if day_first else "bulan/hari/tahun")
        return pd.to_datetime(s, dayfirst=day_first, errors="coerce", format="mixed")

    return pd.to_datetime(s, dayfirst=True, errors="coerce", format="mixed")   # "31 januari 2025", dst


def parse_rate(x):
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = re.sub(r"[^\d.,]", "", str(x))
    if not s:
        return np.nan
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):           # 16.100,50
            s = s.replace(".", "").replace(",", ".")
        else:                                      # 16,100.50
            s = s.replace(",", "")
    elif "," in s:
        s = s.replace(",", "") if re.fullmatch(r"\d{1,3}(,\d{3})+", s) else s.replace(",", ".")
    elif s.count(".") > 1 or re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")                     # 16.100
    return float(s)


def read_jisdor(path):
    if str(path).lower().endswith((".xlsx", ".xls")):
        raw = pd.read_excel(path, header=None)
        # file dari BI kadang punya judul di atas tabel: cari baris yang berisi nama kolom tanggal
        is_hdr = raw.apply(lambda r: r.astype(str).str.strip().str.lower().eq(DATE_COL.lower()).any(), axis=1)
        if is_hdr.any():
            h = is_hdr.idxmax()
            raw.columns = raw.iloc[h].astype(str).str.strip()
            raw = raw.iloc[h + 1:].reset_index(drop=True)
        else:
            raw = pd.read_excel(path)
        return raw
    return pd.read_csv(path, sep=None, engine="python")   # deteksi , atau ; otomatis

raw = read_jisdor(JISDOR_FILE)
missing_cols = [c for c in (DATE_COL, RATE_COL) if c not in raw.columns]
if missing_cols:
    raise KeyError(f"Kolom {missing_cols} tidak ada. Kolom yang terbaca: {list(raw.columns)}")

kurs = pd.DataFrame({
    "date": parse_dates(raw[DATE_COL]),
    "jisdor": raw[RATE_COL].apply(parse_rate),
})
n_raw = len(kurs)
kurs = (kurs.dropna()
            .drop_duplicates("date", keep="last")
            .sort_values("date")
            .reset_index(drop=True))
kurs = kurs[(kurs["date"] >= START_DATE) & (kurs["date"] <= END_DATE)].reset_index(drop=True)

assert kurs["jisdor"].between(5000, 30000).all(), "Ada kurs di luar 5.000-30.000, cek parsing angka"
print(f"Baris mentah: {n_raw}, dipakai: {len(kurs)}")
print(kurs["date"].min().date(), "->", kurs["date"].max().date())
gaps = kurs["date"].diff().dt.days
print("Celah terpanjang antar hari bursa:", gaps.max(), "hari")
print("Tanggal yang didahului celah > 5 hari (kemungkinan data JISDOR bolong):",
      kurs.loc[gaps > 5, "date"].dt.date.tolist())
kurs.head()

## 3. Tempelkan artikel ke hari bursa

Tiap hari bursa punya jam cutoff. Artikel masuk ke hari bursa pertama yang cutoff-nya masih di depan waktu terbitnya. Akibatnya berita Sabtu, Minggu, dan libur nasional otomatis menumpuk ke hari kerja berikutnya (daftar hari bursa diambil dari tanggal di file JISDOR, jadi libur Indonesia ikut tertangani). Berita yang terbit setelah cutoff hari bursa terakhir tidak punya pasangan dan dibuang.

`gap_days` mencatat selisih hari antara tanggal terbit dan hari bursa yang ditempeli, berguna untuk melihat seberapa banyak berita yang menumpuk.

In [ ]:
kurs["cutoff"] = (kurs["date"] + pd.Timedelta(hours=CUTOFF_HOUR, minutes=CUTOFF_MINUTE)
                  ).dt.tz_localize("Asia/Jakarta").astype("datetime64[ns, Asia/Jakarta]")
arts["published_wib"] = arts["published_wib"].astype("datetime64[ns, Asia/Jakarta]")

aligned = pd.merge_asof(
    arts.sort_values("published_wib"),
    kurs[["date", "cutoff"]].rename(columns={"date": "trading_date"}),
    left_on="published_wib", right_on="cutoff", direction="forward",
)

n_before = len(aligned)
aligned = aligned.dropna(subset=["trading_date"]).drop(columns="cutoff").copy()
print(f"Artikel tanpa hari bursa (terbit setelah cutoff terakhir): {n_before - len(aligned)}")

aligned["gap_days"] = (
    aligned["trading_date"] - aligned["published_wib"].dt.tz_localize(None).dt.normalize()
).dt.days
print("\nSebaran gap_days (0 = terbit di hari bursa yang sama):")
print(aligned["gap_days"].value_counts().sort_index())

## 4. Tabel harian: kurs, target, dan fitur berbasis kurs

Definisi target diambil langsung dari `src/jisdor/target_formulation.py` (fungsi `build_target`), jadi sama persis dengan `docs/target_formulation.md` dan `jisdor_target.csv`:

- `target` (**klasifikasi 3-class**): arah pergerakan kurs *besok* terhadap hari ini. `NAIK` kalau naik lebih dari +0,1%, `TURUN` kalau turun lebih dari -0,1%, selain itu `STABIL`. Ambang 0,1% ada di `THRESHOLD_PCT`.
- `kurs_besok` (**regresi**): nilai `kurs_jisdor` hari bursa berikutnya (Jumat -> Senin, bukan hari kalender).
- `pct_change`: perubahan kurs hari ini terhadap kemarin, dalam **persen**. Ini fitur, bukan target.

Kolom tambahan untuk modeling:

- `delta_besok` = `kurs_besok` - `kurs_jisdor`: selisih yang sebenarnya harus ditebak. Persistence sama dengan menebak selisih 0.
- `log_return`, `ret_lag1..3`, `ret_std5`: fitur berbasis kurs. Semuanya hanya memakai informasi sampai hari ini.

Kolom `naik_besok` (biner) dari versi sebelumnya sudah diganti oleh `target`. Baris awal hilang karena lag belum terisi, dan baris terakhir hilang karena tidak punya "besok".

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.jisdor.target_formulation import build_target, THRESHOLD_PCT   # sumber tunggal definisi target tim

base = kurs[["date", "jisdor"]].rename(columns={"date": "tanggal", "jisdor": "kurs_jisdor"})
daily = build_target(base).rename(columns={"tanggal": "date"})   # + pct_change (%), target 3-class, kurs_besok

# fitur berbasis kurs (semua diketahui pada hari t)
daily["log_return"] = np.log(daily["kurs_jisdor"]).diff()
for k in (1, 2, 3):
    daily[f"ret_lag{k}"] = daily["log_return"].shift(k)
daily["ret_std5"] = daily["log_return"].rolling(5).std()
daily["delta_besok"] = daily["kurs_besok"] - daily["kurs_jisdor"]

ties = (kurs["jisdor"].diff() == 0).sum()
daily = daily.dropna().reset_index(drop=True)

# cek 1: kurs_besok hari t sama dengan kurs_jisdor baris berikutnya
assert np.allclose(daily["kurs_besok"].iloc[:-1].values, daily["kurs_jisdor"].iloc[1:].values)
# cek 2: label target konsisten dengan delta_besok dan ambang THRESHOLD_PCT
pct_besok = daily["delta_besok"] / daily["kurs_jisdor"] * 100
expected = np.where(pct_besok > THRESHOLD_PCT, "NAIK", np.where(pct_besok < -THRESHOLD_PCT, "TURUN", "STABIL"))
assert (daily["target"].astype(str).values == expected).all()

print(f"Hari bursa dipakai: {len(daily)} | kurs sama persis dengan kemarin: {ties} hari")
print(f"Ambang STABIL: +-{THRESHOLD_PCT}%")
print(daily["target"].value_counts().to_dict())
daily.head()

## 5. Simpan dan cek cakupan

In [ ]:
aligned["published_wib"] = aligned["published_wib"].astype(str)
aligned.to_csv(PROCESSED_DIR / "articles_aligned.csv", index=False)
daily.to_csv(PROCESSED_DIR / "daily_kurs.csv", index=False)
print("Tersimpan di:", PROCESSED_DIR)

per_day = aligned.groupby("trading_date").size()
covered = daily["date"].isin(per_day.index).mean()
print(f"Hari bursa yang punya minimal 1 artikel: {covered:.1%}")
print(f"Artikel per hari bursa (yang ada berita): median {per_day.median():.0f}, maks {per_day.max()}")